# 🔍 Notebook 05: Exploratory Data Analysis (EDA)

**Objective:** Explore patterns, correlations, and temporal trends in the data.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data.data_loader import load_config, load_csv

config = load_config('../config/config.yaml')
df = load_csv('../' + config['data']['aqi_path'])
print(f"Shape: {df.shape}")

## 5.1 Correlation Heatmap

In [ ]:
core_cols = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3',
             'temperature', 'humidity', 'wind_speed', 'pressure', 'aqi']
core_cols = [c for c in core_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(12, 10))
corr = df[core_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax)
ax.set_title('Correlation Matrix — Pollutants, Weather & AQI', fontsize=14)
plt.tight_layout()
plt.show()

## 5.2 Diurnal Patterns

In [ ]:
df['hour'] = df.index.hour
df['month'] = df.index.month

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
pollutants = ['pm2_5', 'pm10', 'no2', 'so2', 'co', 'o3']

for ax, pol in zip(axes.flat, pollutants):
    if pol in df.columns:
        hourly_mean = df.groupby('hour')[pol].mean()
        ax.plot(hourly_mean.index, hourly_mean.values, 'o-', color='steelblue')
        ax.set_xlabel('Hour of Day')
        ax.set_ylabel(f'{pol.upper()} (µg/m³)')
        ax.set_title(f'{pol.upper()} — Diurnal Pattern')
        ax.grid(True, alpha=0.3)

plt.suptitle('Diurnal Patterns of Pollutants — Delhi', fontsize=14)
plt.tight_layout()
plt.show()

## 5.3 Seasonal/Monthly Patterns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Monthly AQI
monthly_aqi = df.groupby('month')['aqi'].mean()
axes[0].bar(monthly_aqi.index, monthly_aqi.values, color='indianred', edgecolor='white')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Mean AQI')
axes[0].set_title('Monthly Average AQI')
axes[0].set_xticks(range(1, 13))

# Monthly category distribution
cat_order = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
colors = ['#009966', '#FFDE33', '#FF9933', '#CC0033', '#660099', '#7E0023']
monthly_cats = pd.crosstab(df['month'], df['aqi_category'])
monthly_cats = monthly_cats.reindex(columns=cat_order, fill_value=0)
monthly_cats.plot(kind='bar', stacked=True, ax=axes[1], color=colors)
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Count')
axes[1].set_title('AQI Category Distribution by Month')
axes[1].legend(title='Category', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()

## 5.4 Weather vs AQI Relationship

In [ ]:
weather_cols = ['temperature', 'humidity', 'wind_speed', 'pressure']
weather_cols = [c for c in weather_cols if c in df.columns]

if weather_cols:
    fig, axes = plt.subplots(1, len(weather_cols), figsize=(5*len(weather_cols), 5))
    if len(weather_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, weather_cols):
        ax.scatter(df[col], df['aqi'], alpha=0.05, s=1, color='steelblue')
        ax.set_xlabel(col.replace('_', ' ').title())
        ax.set_ylabel('AQI')
        ax.set_title(f'AQI vs {col.replace("_", " ").title()}')
    plt.suptitle('Weather Variables vs AQI', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Clean up temporary columns
df.drop(columns=['hour', 'month'], inplace=True, errors='ignore')
print("EDA complete!")

## ✅ EDA Complete

**Next:** Notebook 06 — Sequence Creation